# 🎨 AIR CANVAS — Finger Tracking Paint
**Draw in the air using your index finger via webcam!**

### Controls:
| Gesture | Action |
|---|---|
| ☝️ Index finger only | Draw |
| ✌️ Index + Middle finger (V) | Stop / Hover (select color) |
| ✊ All fingers closed | Pause drawing |
| Hover over toolbar | Select color / eraser / clear |

Press **`Q`** or **`ESC`** in the canvas window to quit.

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Installing dependencies...')
install('opencv-python')
install('mediapipe')
install('numpy')
print('✅ All dependencies installed!')

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import cv2
import mediapipe as mp
import numpy as np
import time
from collections import deque

print(f'OpenCV  : {cv2.__version__}')
print(f'MediaPipe: {mp.__version__}')
print('✅ Imports OK')

In [ ]:
# ── CELL 3: AIR CANVAS ────────────────────────────────────────────────────────

# ─── Configuration ───────────────────────────────────────────────────────────
CAM_W, CAM_H   = 1280, 720      # Camera resolution (try 640x480 if slow)
SMOOTHING       = 5              # Landmark smoothing window (higher = smoother)
MIN_DETECT_CONF = 0.7
MIN_TRACK_CONF  = 0.6
TOOLBAR_H       = 80             # Height of top toolbar
BRUSH_MIN       = 4
BRUSH_MAX       = 30

# ─── Colour palette ──────────────────────────────────────────────────────────
# (Name, BGR colour, display hex)
PALETTE = [
    ('Red',     (  0,   0, 220), '#DC0000'),
    ('Orange',  (  0, 130, 255), '#FF8200'),
    ('Yellow',  (  0, 220, 255), '#FFDC00'),
    ('Green',   ( 30, 200,  30), '#1EC81E'),
    ('Cyan',    (220, 200,   0), '#00C8DC'),
    ('Blue',    (230,  80,   0), '#0050E6'),
    ('Violet',  (200,  50, 180), '#B432C8'),
    ('Pink',    (180,  80, 255), '#FF50B4'),
    ('White',   (240, 240, 240), '#F0F0F0'),
    ('Eraser',  (  0,   0,   0), '#000000'),   # draws black = erases on black bg
]
ERASER_IDX = len(PALETTE) - 1   # last swatch is eraser

# ─── Helpers ─────────────────────────────────────────────────────────────────
def draw_toolbar(frame, cur_idx, brush_size, canvas_dirty):
    """Render the top toolbar onto frame (in-place)."""
    H, W = frame.shape[:2]
    # semi-transparent background
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, TOOLBAR_H), (15, 15, 20), -1)
    cv2.addWeighted(overlay, 0.85, frame, 0.15, 0, frame)

    # --- colour swatches ---
    n   = len(PALETTE)
    pad = 8
    sw  = (W - (n + 1) * pad) // n   # swatch width
    sw  = min(sw, 60)
    total_w = n * sw + (n + 1) * pad
    start_x = (W - total_w) // 2

    swatch_rects = []
    for i, (name, bgr, _) in enumerate(PALETTE):
        x1 = start_x + pad + i * (sw + pad)
        y1, y2 = pad, TOOLBAR_H - pad
        x2 = x1 + sw
        swatch_rects.append((x1, y1, x2, y2))

        # fill
        if i == ERASER_IDX:
            # eraser: checkerboard
            cv2.rectangle(frame, (x1, y1), (x2, y2), (200, 200, 200), -1)
            cv2.putText(frame, 'ERASE', (x1 + 2, (y1+y2)//2 + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (30, 30, 30), 1, cv2.LINE_AA)
        else:
            cv2.rectangle(frame, (x1, y1), (x2, y2), bgr, -1)

        # border: bright if selected
        border_clr = (0, 255, 180) if i == cur_idx else (80, 80, 80)
        thick      = 3             if i == cur_idx else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), border_clr, thick)

    # --- brush size indicator ---
    bx = W - 120
    by = TOOLBAR_H // 2
    cur_bgr = PALETTE[cur_idx][1] if cur_idx != ERASER_IDX else (200, 200, 200)
    cv2.circle(frame, (bx, by), brush_size // 2, cur_bgr, -1)
    cv2.putText(frame, f'sz:{brush_size}', (bx - 12, by + brush_size // 2 + 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1, cv2.LINE_AA)

    # --- CLEAR button ---
    cx1, cy1, cx2, cy2 = W - 72, pad, W - pad, TOOLBAR_H - pad
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (40, 40, 180), -1)
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (80, 80, 220), 1)
    cv2.putText(frame, 'CLR', (cx1 + 6, (cy1 + cy2)//2 + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 255), 1, cv2.LINE_AA)

    # separator line
    cv2.line(frame, (0, TOOLBAR_H), (W, TOOLBAR_H), (40, 40, 50), 2)

    return swatch_rects, (cx1, cy1, cx2, cy2)


def fingers_up(hand_landmarks, handedness='Right'):
    """
    Returns list [thumb, index, middle, ring, pinky] = 1 if finger is up.
    Works correctly for both hands.
    """
    lm = hand_landmarks.landmark
    tips   = [4, 8, 12, 16, 20]
    joints = [3, 6, 10, 14, 18]
    up = []

    # Thumb: compare x (mirrored feed)
    if handedness == 'Right':
        up.append(1 if lm[4].x < lm[3].x else 0)
    else:
        up.append(1 if lm[4].x > lm[3].x else 0)

    # Other four fingers: tip y < pip y => extended
    for tip, joint in zip(tips[1:], joints[1:]):
        up.append(1 if lm[tip].y < lm[joint].y else 0)

    return up  # [thumb, index, middle, ring, pinky]


class SmoothBuffer:
    """Rolling average for (x, y) coordinates."""
    def __init__(self, size=5):
        self.xq = deque(maxlen=size)
        self.yq = deque(maxlen=size)

    def update(self, x, y):
        self.xq.append(x)
        self.yq.append(y)

    def get(self):
        if not self.xq:
            return None, None
        return int(np.mean(self.xq)), int(np.mean(self.yq))

    def clear(self):
        self.xq.clear()
        self.yq.clear()


# ─── Main loop ───────────────────────────────────────────────────────────────
def run_air_canvas():
    mp_hands   = mp.solutions.hands
    mp_drawing = mp.solutions.drawing_utils
    mp_styles  = mp.solutions.drawing_styles

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
    cap.set(cv2.CAP_PROP_FPS, 30)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # reduce lag

    if not cap.isOpened():
        print('❌ Cannot open camera')
        return

    # Read one frame to get actual size
    ret, frame = cap.read()
    if not ret:
        print('❌ Cannot read frame')
        cap.release()
        return
    H, W = frame.shape[:2]

    # Persistent ink canvas (black background)
    canvas = np.zeros((H, W, 3), dtype=np.uint8)

    # State
    color_idx  = 0
    brush_size = 10
    smoother   = SmoothBuffer(SMOOTHING)
    prev_pt    = None
    drawing    = False
    hover_mode = False       # two-finger pose
    hovered_sw = -1
    hover_start_time = {}
    HOVER_DWELL = 0.9        # seconds to dwell on a swatch to select it

    # FPS counter
    fps_time = time.time()
    fps_val  = 0
    fps_cnt  = 0

    window_name = '🎨 AIR CANVAS  |  Q / ESC = quit'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, min(W, 1280), min(H + TOOLBAR_H, 800))

    # Scroll-wheel → brush size
    def on_mouse(event, x, y, flags, param):
        nonlocal brush_size
        if event == cv2.EVENT_MOUSEWHEEL:
            brush_size = np.clip(brush_size + (2 if flags > 0 else -2),
                                  BRUSH_MIN, BRUSH_MAX)
    cv2.setMouseCallback(window_name, on_mouse)

    with mp_hands.Hands(
        model_complexity      = 1,
        max_num_hands         = 1,
        min_detection_confidence = MIN_DETECT_CONF,
        min_tracking_confidence  = MIN_TRACK_CONF,
    ) as hands:

        print('🎨 AIR CANVAS running — raise your INDEX FINGER to draw!')
        print('   ✌️  Two fingers = hover/select  |  Mouse-wheel = brush size')
        print('   Press Q or ESC in the canvas window to quit.')

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Mirror (selfie view)
            frame = cv2.flip(frame, 1)
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = hands.process(rgb)
            rgb.flags.writeable = True

            # ── Draw toolbar ──────────────────────────────────────────────────
            swatch_rects, clear_rect = draw_toolbar(frame, color_idx, brush_size, True)

            # ── Process hand ─────────────────────────────────────────────────
            tip_pt = None

            if results.multi_hand_landmarks:
                hl = results.multi_hand_landmarks[0]
                hand_label = (
                    results.multi_handedness[0].classification[0].label
                    if results.multi_handedness else 'Right'
                )

                # Draw skeleton
                mp_drawing.draw_landmarks(
                    frame, hl, mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style(),
                )

                up = fingers_up(hl, hand_label)
                index_up  = up[1]
                middle_up = up[2]

                # Index fingertip (landmark 8)
                lm8 = hl.landmark[8]
                raw_x = int(lm8.x * W)
                raw_y = int(lm8.y * H)
                smoother.update(raw_x, raw_y)
                sx, sy = smoother.get()
                tip_pt = (sx, sy)

                # Visual: fingertip dot
                tip_color = PALETTE[color_idx][1] if color_idx != ERASER_IDX else (200, 200, 200)
                cv2.circle(frame, tip_pt, brush_size // 2 + 4, tip_color, 2)
                cv2.circle(frame, tip_pt, 4, (255, 255, 255), -1)

                # ── Gesture detection ─────────────────────────────────────────
                two_fingers = index_up and middle_up   # hover / select mode
                one_finger  = index_up and not middle_up  # draw mode

                if two_fingers:
                    # ── HOVER / SELECT MODE ───────────────────────────────────
                    drawing    = False
                    hover_mode = True
                    smoother.clear()
                    prev_pt = None

                    # Check toolbar collision
                    hit = -1
                    if sy < TOOLBAR_H:
                        for i, (x1, y1, x2, y2) in enumerate(swatch_rects):
                            if x1 <= sx <= x2 and y1 <= sy <= y2:
                                hit = i
                                break
                        # Check CLEAR button
                        cx1, cy1, cx2, cy2 = clear_rect
                        if cx1 <= sx <= cx2 and cy1 <= sy <= cy2:
                            hit = len(PALETTE)  # sentinel for clear

                    # Dwell-to-select logic
                    if hit >= 0:
                        if hit not in hover_start_time:
                            hover_start_time = {hit: time.time()}
                        elapsed = time.time() - hover_start_time.get(hit, time.time())
                        # Draw progress arc on swatch
                        if hit < len(PALETTE):
                            x1, y1, x2, y2 = swatch_rects[hit]
                            cx_s = (x1 + x2) // 2
                            cy_s = (y1 + y2) // 2
                            angle = int(360 * min(elapsed / HOVER_DWELL, 1.0))
                            cv2.ellipse(frame, (cx_s, cy_s), (18, 18), -90,
                                        0, angle, (0, 255, 180), 2)
                        if elapsed >= HOVER_DWELL:
                            if hit == len(PALETTE):   # CLEAR
                                canvas[:] = 0
                            else:
                                color_idx = hit
                            hover_start_time = {}
                    else:
                        hover_start_time = {}

                    hovered_sw = hit
                    # Visual line between index and middle finger
                    lm12 = hl.landmark[12]
                    mid_pt = (int(lm12.x * W), int(lm12.y * H))
                    cv2.line(frame, tip_pt, mid_pt, (0, 255, 180), 2)
                    cv2.putText(frame, 'SELECT', (sx - 28, sy + 22),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 180), 1, cv2.LINE_AA)

                elif one_finger:
                    # ── DRAW MODE ─────────────────────────────────────────────
                    hover_mode = False
                    hover_start_time = {}

                    # Don't draw if finger is in toolbar
                    if sy > TOOLBAR_H:
                        cur_bgr = PALETTE[color_idx][1]
                        eff_size = brush_size * 3 if color_idx == ERASER_IDX else brush_size

                        if drawing and prev_pt:
                            if color_idx == ERASER_IDX:
                                cv2.line(canvas, prev_pt, tip_pt, (0, 0, 0), eff_size)
                            else:
                                cv2.line(canvas, prev_pt, tip_pt, cur_bgr, eff_size)
                        else:
                            drawing = True
                        prev_pt = tip_pt
                    else:
                        prev_pt = None
                        drawing = False

                    cv2.putText(frame, 'DRAW', (sx - 18, sy + 22),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                                PALETTE[color_idx][1], 1, cv2.LINE_AA)
                else:
                    # ── IDLE / FIST ───────────────────────────────────────────
                    drawing = False
                    hover_mode = False
                    prev_pt = None
                    smoother.clear()

            else:
                # No hand detected
                drawing = False
                prev_pt = None
                smoother.clear()

            # ── Composite canvas + camera frame ───────────────────────────────
            # Blend: canvas ink on top of camera feed
            ink_mask = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
            _, ink_mask = cv2.threshold(ink_mask, 5, 255, cv2.THRESH_BINARY)
            inv_mask = cv2.bitwise_not(ink_mask)
            cam_bg  = cv2.bitwise_and(frame, frame, mask=inv_mask)
            ink_fg  = cv2.bitwise_and(canvas, canvas, mask=ink_mask)
            combined = cv2.add(cam_bg, ink_fg)

            # Re-draw toolbar on top of combined
            swatch_rects, clear_rect = draw_toolbar(combined, color_idx, brush_size, True)

            # ── FPS ───────────────────────────────────────────────────────────
            fps_cnt += 1
            if time.time() - fps_time >= 0.5:
                fps_val = fps_cnt * 2
                fps_cnt = 0
                fps_time = time.time()
            cv2.putText(combined, f'FPS:{fps_val}', (10, H - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (80, 80, 80), 1, cv2.LINE_AA)

            # ── HUD instructions ──────────────────────────────────────────────
            hud = [
                '☝ 1 finger = draw',
                '✌ 2 fingers = select',
                '✊ fist = pause',
                'scroll = brush size',
                'Q / ESC = quit',
            ]
            for i, txt in enumerate(hud):
                cv2.putText(combined, txt, (10, TOOLBAR_H + 22 + i * 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100, 100, 100), 1, cv2.LINE_AA)

            cv2.imshow(window_name, combined)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), ord('Q'), 27):   # Q or ESC
                break
            elif key == ord('c') or key == ord('C'):
                canvas[:] = 0
            elif key == ord('s') or key == ord('S'):
                fname = f'air_canvas_{int(time.time())}.png'
                cv2.imwrite(fname, canvas)
                print(f'💾 Saved: {fname}')

    cap.release()
    cv2.destroyAllWindows()
    print('👋 Air Canvas closed.')


# ─── Launch ───────────────────────────────────────────────────────────────────
run_air_canvas()

---
## 🎛️ BONUS — Standalone Ink-Only Mode
Run the cell below for a **plain white canvas** (no camera feed) — useful if webcam is slow or unavailable.

In [ ]:
# ── BONUS CELL: Ink-Only Mode (no camera background) ─────────────────────────
import cv2, mediapipe as mp, numpy as np, time
from collections import deque

TOOLBAR_H = 80
PALETTE = [
    ('Red',    ( 30,  30, 220)),
    ('Orange', (  0, 130, 255)),
    ('Yellow', (  0, 210, 255)),
    ('Green',  ( 40, 180,  40)),
    ('Cyan',   (200, 180,   0)),
    ('Blue',   (220,  60,   0)),
    ('Violet', (180,  40, 160)),
    ('Pink',   (160,  60, 255)),
    ('Black',  ( 10,  10,  10)),
    ('Eraser', (255, 255, 255)),
]
ERASER_IDX = len(PALETTE) - 1
SMOOTH = 5

def run_ink_canvas():
    W, H = 1280, 720
    canvas = np.ones((H, W, 3), dtype=np.uint8) * 255
    mp_h = mp.solutions.hands
    mp_d = mp.solutions.drawing_utils

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, H)

    color_idx = 0; brush = 8
    xq, yq = deque(maxlen=SMOOTH), deque(maxlen=SMOOTH)
    prev = None; drawing = False

    wname = 'INK CANVAS | Q=quit  C=clear  S=save'
    cv2.namedWindow(wname, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(wname, W, H)

    def scroll(ev, x, y, flags, _):
        nonlocal brush
        if ev == cv2.EVENT_MOUSEWHEEL:
            brush = int(np.clip(brush + (2 if flags > 0 else -2), 3, 40))
    cv2.setMouseCallback(wname, scroll)

    with mp_h.Hands(max_num_hands=1, min_detection_confidence=0.7,
                    min_tracking_confidence=0.65) as hands:
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            res = hands.process(rgb)

            display = canvas.copy()

            # toolbar
            cv2.rectangle(display, (0,0), (W, TOOLBAR_H), (240,240,240), -1)
            cv2.line(display, (0, TOOLBAR_H), (W, TOOLBAR_H), (180,180,180), 2)
            n = len(PALETTE); pad = 8
            sw = min((W - (n+1)*pad) // n, 70)
            total_w = n*sw + (n+1)*pad
            sx0 = (W - total_w) // 2
            srects = []
            for i,(name,bgr) in enumerate(PALETTE):
                x1 = sx0 + pad + i*(sw+pad); y1=pad; x2=x1+sw; y2=TOOLBAR_H-pad
                srects.append((x1,y1,x2,y2))
                cv2.rectangle(display,(x1,y1),(x2,y2),bgr,-1)
                if i==ERASER_IDX:
                    cv2.putText(display,'ERASE',(x1+2,(y1+y2)//2+5),
                                cv2.FONT_HERSHEY_SIMPLEX,.35,(80,80,80),1)
                bc = (0,200,100) if i==color_idx else (150,150,150)
                bk = 3 if i==color_idx else 1
                cv2.rectangle(display,(x1,y1),(x2,y2),bc,bk)
            # clear btn
            cv2.rectangle(display,(W-70,pad),(W-pad,TOOLBAR_H-pad),(220,80,80),-1)
            cv2.putText(display,'CLR',(W-62,(TOOLBAR_H)//2+5),
                        cv2.FONT_HERSHEY_SIMPLEX,.45,(255,255,255),1)

            if res.multi_hand_landmarks:
                hl = res.multi_hand_landmarks[0]
                lm = hl.landmark
                idx_up = lm[8].y < lm[6].y
                mid_up = lm[12].y < lm[10].y
                tx = int(lm[8].x * W); ty = int(lm[8].y * H)
                xq.append(tx); yq.append(ty)
                sx = int(np.mean(xq)); sy = int(np.mean(yq))

                col = PALETTE[color_idx][1]
                cv2.circle(display,(sx,sy),brush//2+3,col,2)

                if idx_up and not mid_up:
                    if sy > TOOLBAR_H:
                        ec = (255,255,255) if color_idx==ERASER_IDX else col
                        bsz = brush*3 if color_idx==ERASER_IDX else brush
                        if drawing and prev:
                            cv2.line(canvas,prev,(sx,sy),ec,bsz)
                        drawing=True; prev=(sx,sy)
                    else:
                        drawing=False; prev=None
                elif idx_up and mid_up:
                    drawing=False; prev=None; xq.clear(); yq.clear()
                    if sy<TOOLBAR_H:
                        for i,(x1,y1,x2,y2) in enumerate(srects):
                            if x1<=sx<=x2: color_idx=i; break
                        if W-70<=sx<=W-pad: canvas[:]=255
                else:
                    drawing=False; prev=None; xq.clear(); yq.clear()

            cv2.imshow(wname, display)
            k = cv2.waitKey(1) & 0xFF
            if k in (ord('q'),27): break
            elif k==ord('c'): canvas[:]=255
            elif k==ord('s'):
                f=f'ink_{int(time.time())}.png'; cv2.imwrite(f,canvas); print(f'💾 {f}')

    cap.release(); cv2.destroyAllWindows(); print('Done.')

run_ink_canvas()

---
## 📖 Quick Reference

| Key | Action |
|---|---|
| `Q` / `ESC` | Quit |
| `C` | Clear canvas |
| `S` | Save canvas as PNG |
| Mouse scroll | Increase / decrease brush size |

| Gesture | Action |
|---|---|
| ☝️ Index finger up | Draw |
| ✌️ Index + Middle up | Hover / select colour from toolbar |
| ✊ Fist / all down | Pause (lift pen) |

### Tips for best accuracy
- Use in **good lighting** — avoid backlit setups
- Keep your **hand fully visible** in frame
- Slow, deliberate strokes are smoother than fast flicks
- Adjust `SMOOTHING` (Cell 3, top) — higher = smoother but slightly laggy
- Adjust `MIN_DETECT_CONF` / `MIN_TRACK_CONF` if hand detection is flaky